# AlphaMissense vs ClinVar - step-by-step benchmark

Run **one cell at a time**. We treat ClinVar clinical labels as the ground truth
and ask: **how well does the AlphaMissense pathogenicity score separate
pathogenic from benign missense variants?**

> The data file `data/variants.csv` is produced by `fetch_variants.py`, which pulls
> variants that have BOTH an AlphaMissense score and a ClinVar label from
> MyVariant.info. Run that script first if the file is missing.

> Launch from the project folder: `cd alphamissense_clinvar_benchmark && .venv/bin/jupyter lab`

## 1. Load the variants

In [ ]:
import pandas as pd

df = pd.read_csv('data/variants.csv')
print('total variants:', len(df))
df['clinvar_significance'].value_counts().head(12)

## 2. Turn ClinVar labels into a binary target

Pathogenic / Likely pathogenic -> 1, Benign / Likely benign -> 0. We drop
'uncertain' and 'conflicting' (no ground truth) and keep only variants that
actually have a numeric AlphaMissense score (i.e. real missense variants).

In [ ]:
pathogenic_labels = ['Pathogenic', 'Likely pathogenic', 'Pathogenic/Likely pathogenic']
benign_labels = ['Benign', 'Likely benign', 'Benign/Likely benign']

labels = []
for sig in df['clinvar_significance']:
    if sig in pathogenic_labels:
        labels.append(1)
    elif sig in benign_labels:
        labels.append(0)
    else:
        labels.append(None)
df['label'] = labels

df['am_score'] = pd.to_numeric(df['am_score'], errors='coerce')
df = df.dropna(subset=['label', 'am_score'])
df['label'] = df['label'].astype(int)

print('labelled missense variants:', len(df))
print('pathogenic:', int(df['label'].sum()), ' benign:', int((df['label'] == 0).sum()))

## 3. Benchmark: does the AlphaMissense score separate the two?

ROC-AUC asks how well the score ranks pathogenic above benign (0.5 = random,
1.0 = perfect). Average precision is a second, imbalance-aware summary.

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

auc = roc_auc_score(df['label'], df['am_score'])
ap = average_precision_score(df['label'], df['am_score'])
print('ROC-AUC          : %.3f' % auc)
print('Average precision: %.3f' % ap)

## 4. ROC curve

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve

fpr, tpr, thresholds = roc_curve(df['label'], df['am_score'])

plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, label='AlphaMissense (AUC = %.3f)' % auc)
plt.plot([0, 1], [0, 1], '--', color='grey', label='random')
plt.xlabel('false positive rate')
plt.ylabel('true positive rate')
plt.title('AlphaMissense vs ClinVar - ROC curve')
plt.legend(loc='lower right')
plt.show()

## 5. AlphaMissense's own call vs ClinVar

AlphaMissense also gives a class: P (likely pathogenic), B (likely benign),
A (ambiguous). How often does its P/B call agree with ClinVar?

In [ ]:
called = df[df['am_pred'].isin(['P', 'B'])]
correct = 0
for pred, label in zip(called['am_pred'], called['label']):
    if (pred == 'P' and label == 1) or (pred == 'B' and label == 0):
        correct += 1

print('variants called P or B :', len(called))
print('agreement with ClinVar : %.3f' % (correct / len(called)))
print('called ambiguous       :', int((df['am_pred'] == 'A').sum()))

## 6. Spotlight - NPC1 I1061T (the paper's 'I1063T')

The most common Niemann-Pick C1 mutation. ClinVar calls it pathogenic - see what
AlphaMissense says. (A good reminder that a high overall AUC can still hide
misses on individual, important variants.)

In [ ]:
spot = df[df['variant_id'] == 'chr18:g.21116700A>G']
spot[['gene', 'protein_change', 'clinvar_significance', 'am_score', 'am_pred']]